In [6]:
import pandas as pd

In [7]:
!pip install pymysql

In [8]:
import boto3
from io import StringIO

s3 = boto3.client("s3", region_name="us-east-1")
bucket = "food-waste-project"

def load_csv(key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))

waste = load_csv("predictions/waste_predictions_rds.csv")
reorder = load_csv("predictions/reorder_predictions.csv")
spoilage = load_csv("predictions/spoilage_predictions.csv")

print("Waste shape:", waste.shape)
print("Reorder shape:", reorder.shape)
print("Spoilage shape:", spoilage.shape)

Waste shape: (36143, 5)
Reorder shape: (16760, 3)
Spoilage shape: (36143, 4)


In [9]:
#merge everything
final_df = waste.merge(reorder, on=["product_id", "product_name"], how="inner")
final_df = final_df.merge(spoilage, on=["product_id", "product_name"], how="inner")


In [10]:
# numeric mappings
spoilage_map = {
    "high": 1.0,
    "medium": 0.5,
    "low": 0.1
}

waste_map = {
    "high": 1.0,
    "medium": 0.5,
    "low": 0.1
}

# apply mappings
final_df["spoilage_score"] = final_df["spoilage_risk"].map(spoilage_map)
final_df["waste_score"] = final_df["waste_risk"].map(waste_map)

In [11]:
# Final score
final_df["final_score"] = (
    final_df["reorder_probability"]
    - 0.25 * final_df["spoilage_score"]
    - 0.20 * final_df["waste_score"]
)

In [12]:
# Sort
final_df = final_df.sort_values(by="final_score", ascending=False)

In [13]:
# Save for app
final_df.to_csv("final_grocery_list.csv", index=False)

print(final_df.head(20))

       product_id                                       product_name  \
494          1870                         Chipotle Black Bean Burger   
9261        32945        Renola Cocoa Coconut Fruit Nut and Seed Mix   
481          1820                             Organic Fusilli No. 34   
11512       40775                                   Organic Linguine   
2056         7398                                 Indian Samosa Wrap   
12143       43152                                        Mini Bagels   
1522         5467              Spinach & Cheese Enchilada Verde Meal   
11194       39683                            Organic Quinoa Macaroni   
12526       44536              Simple Favorites Five Cheese Rigatoni   
11001       39005                   100% Whole Wheat Farfalle No. 87   
2951        10560                Cheerios Toasted Whole Grain Cereal   
5982        21243  Simple Favorites Chicken Fettuccini Frozen Entree   
3885        13866                                Deluxe Plain Ba

Our system prioritizes items that are likely to be reordered while penalizing products that are more prone to waste or spoilage.

In [15]:
import io

# Save to S3
csv_buffer = io.StringIO()
final_df.to_csv(csv_buffer, index=False)
s3.put_object(
    Bucket=bucket,
    Key="predictions/final_grocery_list.csv",
    Body=csv_buffer.getvalue()
)
print("Saved to S3!")

Saved to S3!


We built a system that recommends grocery items by balancing user demand with spoilage and waste risk, helping reduce unnecessary food waste

this output is a ranked grocery list from best to worst purchases

Instead of recommending everything users might buy, we prioritize items they will, reducing waste and improving grocery decisions.”

In [16]:
import pymysql

conn = pymysql.connect(
    host="food-waste-db.cxkacdkaxalv.us-east-1.rds.amazonaws.com",
    user="admin",
    password="ChurnLab2026!",
    port=3306,
    database="food_waste"
)

cursor = conn.cursor()

# Create table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS final_grocery_list (
        product_id INT,
        product_name TEXT,
        predicted_waste_category TEXT,
        waste_risk TEXT,
        reorder_probability FLOAT,
        spoilage_risk TEXT,
        spoilage_score FLOAT,
        waste_score FLOAT,
        final_score FLOAT
    );
""")

# Upload
for _, row in final_df.iterrows():
    cursor.execute("""
        INSERT INTO final_grocery_list
        (product_id, product_name, predicted_waste_category, waste_risk, 
         reorder_probability, spoilage_risk, spoilage_score, waste_score, final_score)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row["product_id"], row["product_name"], row["predicted_waste_category"],
        row["waste_risk"], row["reorder_probability"], row["spoilage_risk"],
        row["spoilage_score"], row["waste_score"], row["final_score"]
    ))

conn.commit()
cursor.close()
conn.close()
print("Final grocery list uploaded to RDS!")

Final grocery list uploaded to RDS!


In [ ]:
import pymysql
import pandas as pd

conn = pymysql.connect(
    host="food-waste-db.cxkacdkaxalv.us-east-1.rds.amazonaws.com",
    user="admin", password="ChurnLab2026!", port=3306, database="food_waste"
)

# Check how many products match across all tables
query = """
SELECT COUNT(*) as matching_products
FROM final_grocery_list f
INNER JOIN waste_predictions w ON f.product_id = w.product_id
INNER JOIN reorder_predictions r ON f.product_id = r.product_id
INNER JOIN spoilage_predictions s ON f.product_id = s.product_id;
"""

result = pd.read_sql(query, conn)
print("Products matching across all tables:", result)

# Also check row counts per table
for table in ["final_grocery_list", "waste_predictions", "reorder_predictions", "spoilage_predictions"]:
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    print(f"{table}: {cursor.fetchone()[0]} rows")

conn.close()

In [ ]:
#create random grocery list for tableau demo

In [17]:
import pymysql
import pandas as pd
from io import StringIO
import boto3

# Load final grocery list from S3
s3 = boto3.client("s3", region_name="us-east-1")
obj = s3.get_object(Bucket="food-waste-project", Key="predictions/final_grocery_list.csv")
final_df = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))

# Create random sample of 20 products
random_sample = final_df.sample(n=20, random_state=None)  # random_state=None means truly random each time

print(random_sample[["product_name", "final_score", "waste_risk"]].head(20))

# Upload to RDS as its own table
conn = pymysql.connect(
    host="food-waste-db.cxkacdkaxalv.us-east-1.rds.amazonaws.com",
    user="admin", password="ChurnLab2026!", port=3306, database="food_waste"
)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS random_grocery_list (
        product_id INT,
        product_name TEXT,
        predicted_waste_category TEXT,
        waste_risk TEXT,
        reorder_probability FLOAT,
        spoilage_risk TEXT,
        final_score FLOAT
    );
""")

# Clear old random sample first
cursor.execute("DELETE FROM random_grocery_list;")

for _, row in random_sample.iterrows():
    cursor.execute("""
        INSERT INTO random_grocery_list
        (product_id, product_name, predicted_waste_category, waste_risk,
         reorder_probability, spoilage_risk, final_score)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (row["product_id"], row["product_name"], row["predicted_waste_category"],
          row["waste_risk"], row["reorder_probability"], row["spoilage_risk"],
          row["final_score"]))

conn.commit()
cursor.close()
conn.close()
print("Random grocery list uploaded to RDS!")

                                            product_name  final_score  \
4378                          Organic Fruitful Os Cereal        0.405   
6868   Steamfresh Chef's Favorites Lightly Sauced Roa...        0.327   
10989                     Bernie's Farm Cheddar Crackers        0.174   
2311             Old World Style Traditional Pasta Sauce        0.476   
13839    Jimmy Dean Fully Cooked Turkey Sausage Crumbles       -0.085   
3062                              Vanilla Bean Ice Cream        0.450   
620                        Protein Power Greens Smoothie        0.566   
2425                                             Limeade        0.472   
5219                          Polenta Enriched Corn Meal        0.378   
198                Organic California Brown Basmati Rice        0.620   
7081                Mrs Fields Cookie Sandwich Ice Cream        0.320   
8734   Multigrain Blue Tortilla Chips With Sea Salt &...        0.261   
1058                                  Texas Garlic 

In [ ]:
# grocery planning

In [19]:
import pymysql
import boto3
import pandas as pd
from io import StringIO

# Load final grocery list from S3
s3 = boto3.client("s3", region_name="us-east-1")
obj = s3.get_object(Bucket="food-waste-project", Key="predictions/final_grocery_list.csv")
final_df = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))

# Add estimated servings per product category
def estimate_servings(row):
    dept = str(row.get("predicted_waste_category", "")).lower()
    if dept == "fresh":
        return 4
    elif dept == "frozen":
        return 6
    elif dept == "pantry":
        return 8
    elif dept == "beverage":
        return 6
    elif dept == "snack":
        return 4
    else:
        return 4

final_df["servings_per_unit"] = final_df.apply(estimate_servings, axis=1)

# Connect to RDS
conn = pymysql.connect(
    host="food-waste-db.cxkacdkaxalv.us-east-1.rds.amazonaws.com",
    user="admin",
    password="ChurnLab2026!",
    port=3306,
    database="food_waste"
)

cursor = conn.cursor()

# Add new column
try:
    cursor.execute("""
        ALTER TABLE final_grocery_list 
        ADD COLUMN servings_per_unit INT DEFAULT 4;
    """)
    print("Column added!")
except Exception as e:
    print("Column may already exist, skipping:", e)

# Update values
for _, row in final_df.iterrows():
    cursor.execute("""
        UPDATE final_grocery_list 
        SET servings_per_unit = %s 
        WHERE product_id = %s
    """, (row["servings_per_unit"], row["product_id"]))

conn.commit()
cursor.close()
conn.close()
print("Servings data uploaded!")

Column added!
Servings data uploaded!


In [ ]:
#Random Optimized List

In [21]:
import pymysql
import boto3
import pandas as pd
from io import StringIO

# Load final grocery list from S3
s3 = boto3.client("s3", region_name="us-east-1")
obj = s3.get_object(Bucket="food-waste-project", Key="predictions/final_grocery_list.csv")
final_df = pd.read_csv(StringIO(obj["Body"].read().decode("utf-8")))

# Add servings per unit
def estimate_servings(row):
    dept = str(row.get("predicted_waste_category", "")).lower()
    if dept == "fresh":
        return 4
    elif dept == "frozen":
        return 6
    elif dept == "pantry":
        return 8
    elif dept == "beverage":
        return 6
    elif dept == "snack":
        return 4
    else:
        return 4

final_df["servings_per_unit"] = final_df.apply(estimate_servings, axis=1)

# Take a random sample balanced across categories
random_optimized = final_df.groupby("predicted_waste_category").apply(
    lambda x: x.sample(min(len(x), 5), random_state=None)
).reset_index(drop=True)

# Sort by final score
random_optimized = random_optimized.sort_values("final_score", ascending=False)

print("Sample:")
print(random_optimized[["product_name", "waste_risk", "final_score"]].head(20))

# Upload to RDS
conn = pymysql.connect(
    host="food-waste-db.cxkacdkaxalv.us-east-1.rds.amazonaws.com",
    user="admin", password="ChurnLab2026!", port=3306, database="food_waste"
)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS random_optimized_list (
        product_id INT,
        product_name TEXT,
        predicted_waste_category TEXT,
        waste_risk TEXT,
        reorder_probability FLOAT,
        spoilage_risk TEXT,
        spoilage_score FLOAT,
        waste_score FLOAT,
        final_score FLOAT,
        servings_per_unit INT
    );
""")

cursor.execute("DELETE FROM random_optimized_list;")

for _, row in random_optimized.iterrows():
    cursor.execute("""
        INSERT INTO random_optimized_list
        (product_id, product_name, predicted_waste_category, waste_risk,
         reorder_probability, spoilage_risk, spoilage_score, waste_score, 
         final_score, servings_per_unit)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row["product_id"], row["product_name"], row["predicted_waste_category"],
        row["waste_risk"], row["reorder_probability"], row["spoilage_risk"],
        row["spoilage_score"], row["waste_score"], row["final_score"],
        row["servings_per_unit"]
    ))

conn.commit()
cursor.close()
conn.close()
print("Random optimized list uploaded to RDS!")

Sample:
                                         product_name waste_risk  final_score
17                  Provolone & Prosciutto Tortellini        low        0.593
12  Organic Gluten Free Non-Dairy Beans & Rice Bur...        low        0.584
10        Brown Rice Black-Eyed Peas and Veggies Bowl        low        0.544
18    Organic Light in Sodium Low-Fat Minestrone Soup        low        0.522
21       Medleys Assorted Fruit Fruit Flavored Snacks     medium        0.411
1                Organic Strawberry Hibiscus Kombucha     medium        0.389
0                                          Pinot Noir     medium        0.388
23                     Organic Himalayan Pink Popcorn     medium        0.382
4                  Decaf House Blend K-Cup Deep Roast     medium        0.376
16              Premium Cannellini White Kidney Beans        low        0.356
19                                     Sherry Vinegar        low        0.324
14                 Top Chef Crustless Chicken Pot Pie   

/tmp/ipykernel_14697/2441249532.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  random_optimized = final_df.groupby("predicted_waste_category").apply(
